In [1]:
import pandas as pd
from datasets import load_dataset

# 1. Load dataset from your cache
ds = load_dataset("HIDIVE/DQVis")
df = ds["train"].to_pandas()

# 2. Check all 15 column names and remaining categories
print("=== 1. ALL 15 COLUMNS ===")
print(df.columns.tolist())

for col in df.columns:
    if col not in ["query", "query_base", "spec", "expertise", "formality", "spec_key_count"]:
        if df[col].nunique() < 25:
            print(f"\n--- {col} ---")
            print(df[col].value_counts(dropna=False))

# 3. Check the 'expertise == 6' outlier (formality stops at 5)
print("\n=== 2. EXPERTISE == 6 OUTLIERS ===")
exp6 = df[df["expertise"] == 6]
print(f"Total rows with expertise == 6: {len(exp6)}")
for _, row in exp6.head(3).iterrows():
    print(f"Chart: {row['chart_type']} | Formality: {row['formality']} | Query: {row['query']}")

# 4. Check tiny/incomplete specs (spec_key_count <= 4)
print("\n=== 3. MINIMAL SPECS (<= 4 KEYS) ===")
tiny = df[df["spec_key_count"] <= 4]
print(f"Total rows with <= 4 keys: {len(tiny)}")
for _, row in tiny.head(3).iterrows():
    print(f"Keys: {row['spec_key_count']} | Chart: {row['chart_type']} | Query: {row['query']}")
    print(f"Full Spec: {row['spec']}\n")

# 5. Find rows where the query asks for one chart, but chart_type says another
print("=== 4. EXPLICIT CHART TYPE MISMATCHES ===")
bar_bad = df[df["query"].str.contains(r"\bbar chart\b", case=False, na=False) & ~df["chart_type"].str.contains("bar", case=False, na=False)]
line_bad = df[df["query"].str.contains(r"\bline chart\b|\bline graph\b", case=False, na=False) & ~df["chart_type"].str.contains("line", case=False, na=False)]
scatter_bad = df[df["query"].str.contains(r"\bscatter\b", case=False, na=False) & ~df["chart_type"].str.contains("scatter", case=False, na=False)]

print(f"Bar chart mismatches:    {len(bar_bad)}")
print(f"Line chart mismatches:   {len(line_bad)}")
print(f"Scatter plot mismatches: {len(scatter_bad)}")

for label, subset in [("BAR MISMATCH", bar_bad), ("LINE MISMATCH", line_bad), ("SCATTER MISMATCH", scatter_bad)]:
    for _, row in subset.head(2).iterrows():
        print(f"\n[{label}] Assigned chart_type: {row['chart_type']}")
        print(f"Natural Query : {row['query']}")
        print(f"Base Query    : {row['query_base']}")
        print(f"Full Spec     : {row['spec']}")

/Users/siwon/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/siwon/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


=== 1. ALL 15 COLUMNS ===
['query', 'expertise', 'formality', 'query_template', 'constraints', 'spec_template', 'query_type', 'creation_method', 'chart_type', 'chart_complexity', 'spec_key_count', 'query_base', 'spec', 'solution', 'dataset_schema']

--- query_type ---
query_type
question     952359
utterance    122831
Name: count, dtype: int64

--- creation_method ---
creation_method
template    1075190
Name: count, dtype: int64

--- chart_type ---
chart_type
heatmap            335373
stacked_bar        250302
barchart           143562
grouped_scatter    137440
table              129684
grouped_line        23537
grouped_dot         21859
scatterplot         18483
circular             5000
grouped_area         4275
line                 2100
histogram            1825
dot                  1350
area                  400
Name: count, dtype: int64

--- chart_complexity ---
chart_complexity
medium           522782
simple           329905
complex          179202
extra complex     43301
Name: c